# Analítica del Marketing | Práctica de A/B Testing
## Análisis del experimento Cookie Cats

**Objetivo:** Analizar el efecto de mover la primera puerta del juego del nivel 30 al nivel 40 sobre el comportamiento y retención de los jugadores, comparando pruebas A/B con modelos de regresión.

---
## Importación de librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

from scipy import stats
from scipy.stats import mannwhitneyu, chi2_contingency, ttest_ind
from statsmodels.stats.proportion import proportions_ztest
import statsmodels.formula.api as smf
import statsmodels.api as sm
import warnings
warnings.filterwarnings('ignore')

# Estilo de gráficos
sns.set_theme(style='whitegrid', palette='muted')
COLORS = {'gate_30': '#4C72B0', 'gate_40': '#DD8452'}

print('Librerías cargadas correctamente ✓')

---
## Carga y preparación de datos

In [ ]:
# ── Carga el archivo  ──
df = pd.read_csv('cookie_cats.csv')

print('=== Primeras filas ===')
display(df.head())

print(f'\n=== Dimensiones ===')
print(f'Filas: {df.shape[0]:,}  |  Columnas: {df.shape[1]}')

print('\n=== Tipos de datos ===')
print(df.dtypes)

In [ ]:
# ── Verificar grupos del experimento ──────────────────────────────────────────
print('=== Distribución por versión ===')
print(df['version'].value_counts())
print(f'\nTotal usuarios: {len(df):,}')

# Verificar valores nulos
print('\n=== Valores nulos ===')
print(df.isnull().sum())

In [ ]:
# ── Preparación de variables ───────────────────────────────────────────────────

# Recodificar retention_1 y retention_7 como enteros binarios (0/1)
df['retention_1'] = df['retention_1'].astype(int)
df['retention_7'] = df['retention_7'].astype(int)

# Convertir version a variable categórica con gate_30 como referencia
df['version'] = pd.Categorical(df['version'], categories=['gate_30', 'gate_40'], ordered=False)

# Variable dummy para regresión (gate_40 = 1, gate_30 = 0)
df['gate_40'] = (df['version'] == 'gate_40').astype(int)

print('Variables preparadas:')
print(df[['version', 'retention_1', 'retention_7', 'gate_40']].dtypes)
print('\nEjemplo de registros:')
display(df[['userid', 'version', 'sum_gamerounds', 'retention_1', 'retention_7', 'gate_40']].head(8))

---
## Exploración de datos

In [ ]:
# ── Estadísticas descriptivas ──────
desc = df.groupby('version')['sum_gamerounds'].describe().round(2)
print('=== Estadísticas descriptivas: sum_gamerounds ===')
display(desc)

print('\n=== Mediana por grupo ===')
print(df.groupby('version')['sum_gamerounds'].median())

In [ ]:
# ── Tasas de retención por grupo ───────────────────────────────────────────────
retention = df.groupby('version')[['retention_1', 'retention_7']].mean().round(4) * 100
retention.columns = ['Retención día 1 (%)', 'Retención día 7 (%)']
print('=== Tasas de retención por versión ===')
display(retention)

# Diferencia entre grupos
diff = retention.loc['gate_40'] - retention.loc['gate_30']
print('\n=== Diferencia gate_40 - gate_30 (puntos porcentuales) ===')
print(diff.round(3))

In [ ]:
# ── Visualizaciones ─────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Exploración visual: Cookie Cats A/B Test', fontsize=14, fontweight='bold', y=1.02)

# --- Gráfico 1: Boxplot sum_gamerounds (sin outliers extremos para visualizar mejor)
ax1 = axes[0]
cap = df['sum_gamerounds'].quantile(0.99)
df_viz = df[df['sum_gamerounds'] <= cap]
sns.boxplot(data=df_viz, x='version', y='sum_gamerounds',
            palette=COLORS, ax=ax1, width=0.5)
ax1.set_title('Rondas jugadas por grupo\n(sin top 1% outliers)', fontweight='bold')
ax1.set_xlabel('Versión')
ax1.set_ylabel('sum_gamerounds')

# --- Gráfico 2: Barras retención día 1
ax2 = axes[1]
ret1 = df.groupby('version')['retention_1'].mean() * 100
bars = ax2.bar(ret1.index, ret1.values, color=[COLORS[v] for v in ret1.index], width=0.5, edgecolor='white')
ax2.set_title('Tasa de retención\nDía 1', fontweight='bold')
ax2.set_ylabel('Retención (%)')
ax2.set_ylim(0, ret1.max() * 1.2)
for bar, val in zip(bars, ret1.values):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
             f'{val:.2f}%', ha='center', va='bottom', fontweight='bold')

# --- Gráfico 3: Barras retención día 7
ax3 = axes[2]
ret7 = df.groupby('version')['retention_7'].mean() * 100
bars = ax3.bar(ret7.index, ret7.values, color=[COLORS[v] for v in ret7.index], width=0.5, edgecolor='white')
ax3.set_title('Tasa de retención\nDía 7', fontweight='bold')
ax3.set_ylabel('Retención (%)')
ax3.set_ylim(0, ret7.max() * 1.2)
for bar, val in zip(bars, ret7.values):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
             f'{val:.2f}%', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig('exploracion_visual.png', dpi=150, bbox_inches='tight')
plt.show()
print('\nObservación: No existen diferencias visibles entre gate_30 y gate_40')

In [ ]:
# ── Distribución de rondas jugadas (histograma) ────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))
cap = df['sum_gamerounds'].quantile(0.95)
for version, color in COLORS.items():
    subset = df[df['version'] == version]['sum_gamerounds']
    subset[subset <= cap].hist(bins=40, alpha=0.6, color=color, label=version, ax=ax, density=True)
ax.set_title('Distribución de rondas jugadas por grupo (hasta percentil 95)', fontweight='bold')
ax.set_xlabel('sum_gamerounds')
ax.set_ylabel('Densidad')
ax.legend()
plt.tight_layout()
plt.savefig('distribucion_rondas.png', dpi=150, bbox_inches='tight')
plt.show()

**Comentario:**  Los boxplots son similares  en la mediana y Las tasas de retención no difieren entre grupos*

---
## Pruebas A/B

### Prueba para sum_gamerounds

Dado que `sum_gamerounds` es una variable numérica muy sesgada (muchos usuarios con pocas rondas y pocos con muchas), se aplica primero una prueba t de Student como referencia, y complementariamente la prueba de Mann-Whitney U (no paramétrica) como alternativa más robusta.

In [ ]:
# ── Separar grupos ───
g30_rounds = df[df['version'] == 'gate_30']['sum_gamerounds']
g40_rounds = df[df['version'] == 'gate_40']['sum_gamerounds']

# ── Verificar normalidad (Shapiro en muestra) ──────────────────────────────────
# Con n muy grande, Shapiro no es práctico; usamos histograma para justificar
print('Asimetría (skewness):')
print(f'  gate_30: {g30_rounds.skew():.2f}')
print(f'  gate_40: {g40_rounds.skew():.2f}')
print('→ Alta asimetría positiva: la distribución no es normal.')
print('  Se usará Mann-Whitney U como prueba principal (no paramétrica).')

In [ ]:
# ── Prueba t de Student  ──────────────────
t_stat, t_pval = ttest_ind(g30_rounds, g40_rounds, equal_var=False)

print('=' * 55)
print('PRUEBA A/B 1: sum_gamerounds — Prueba t de Welch')
print('=' * 55)
print('H₀: μ(gate_30) = μ(gate_40)  [no hay diferencia en medias]')
print('H₁: μ(gate_30) ≠ μ(gate_40)  [hay diferencia en medias]')
print(f'\nEstadístico t : {t_stat:.4f}')
print(f'Valor p       : {t_pval:.4f}')
alpha = 0.05
conclusion = 'Se rechaza H₀' if t_pval < alpha else 'No se rechaza H₀'
print(f'Conclusión    : {conclusion} (α = {alpha})')

print('\nMedias:')
print(f'  gate_30: {g30_rounds.mean():.2f}')
print(f'  gate_40: {g40_rounds.mean():.2f}')

In [ ]:
# ── Mann-Whitney U ─────────
u_stat, u_pval = mannwhitneyu(g30_rounds, g40_rounds, alternative='two-sided')

print('=' * 55)
print('PRUEBA A/B 1b: sum_gamerounds — Mann-Whitney U')
print('=' * 55)
print('H₀: La distribución de rondas es igual en ambos grupos.')
print('H₁: La distribución de rondas difiere entre grupos.')
print(f'\nEstadístico U : {u_stat:.0f}')
print(f'Valor p       : {u_pval:.4f}')
conclusion = 'Se rechaza H₀' if u_pval < alpha else 'No se rechaza H₀'
print(f'Conclusión    : {conclusion} (α = {alpha})')

print('\nMedianas:')
print(f'  gate_30: {g30_rounds.median():.1f}')
print(f'  gate_40: {g40_rounds.median():.1f}')

**Interpretación:** significa que no hay evidencia estadísticamente significativa de que la distribución del número de rondas jugadas sea diferente entre los dos grupos (gate_30 y gate_40).Aunque las medianas son muy cercanas (17.0 vs 16.0), la diferencia observada no es estadísticamente significativa.






### Prueba para retention_1

In [ ]:
# ── Tabla de contingencia ──────────────────────────────────────────────────────
ct1 = pd.crosstab(df['version'], df['retention_1'])
print('Tabla de contingencia: version × retention_1')
display(ct1)

# ── Prueba chi-cuadrada ────────────────────────────────────────────────────────
chi2_1, p_chi2_1, dof_1, expected_1 = chi2_contingency(ct1)

# ── Prueba Z de proporciones ───────────────────────────────────────────────────
counts1 = np.array([df[df['version']=='gate_30']['retention_1'].sum(),
                    df[df['version']=='gate_40']['retention_1'].sum()])
nobs1   = np.array([len(df[df['version']=='gate_30']),
                    len(df[df['version']=='gate_40'])])
z1, p_z1 = proportions_ztest(counts1, nobs1)

print('\n' + '=' * 55)
print('PRUEBA A/B 2: retention_1')
print('=' * 55)
print('H₀: p(ret1 | gate_30) = p(ret1 | gate_40)')
print('H₁: p(ret1 | gate_30) ≠ p(ret1 | gate_40)')
print(f'\n[Chi-cuadrada]')
print(f'  χ² = {chi2_1:.4f}, gl = {dof_1}, p = {p_chi2_1:.4f}')
print(f'[Prueba Z de proporciones]')
print(f'  z  = {z1:.4f},            p = {p_z1:.4f}')
conclusion = 'Se rechaza H₀' if p_chi2_1 < alpha else 'No se rechaza H₀'
print(f'\nConclusión: {conclusion} (α = {alpha})')
print(f'\nTasas de retención día 1:')
print(f'  gate_30: {counts1[0]/nobs1[0]*100:.2f}%')
print(f'  gate_40: {counts1[1]/nobs1[1]*100:.2f}%')

**Interpretación:** No se rechaza la hipótesis nula (H₀).  No existe evidencia estadísticamente significativa de que haya una diferencia en la tasa de retención al día 1 entre los grupos gate_30 y gate_40.



### Prueba para retention_7

In [ ]:
# ── Tabla de contingencia ──────────────────────────────────────────────────────
ct7 = pd.crosstab(df['version'], df['retention_7'])
print('Tabla de contingencia: version × retention_7')
display(ct7)

# ── Prueba chi-cuadrada ────────────────────────────────────────────────────────
chi2_7, p_chi2_7, dof_7, expected_7 = chi2_contingency(ct7)

# ── Prueba Z de proporciones ───────────────────────────────────────────────────
counts7 = np.array([df[df['version']=='gate_30']['retention_7'].sum(),
                    df[df['version']=='gate_40']['retention_7'].sum()])
nobs7   = np.array([len(df[df['version']=='gate_30']),
                    len(df[df['version']=='gate_40'])])
z7, p_z7 = proportions_ztest(counts7, nobs7)

print('\n' + '=' * 55)
print('PRUEBA A/B 3: retention_7')
print('=' * 55)
print('H₀: p(ret7 | gate_30) = p(ret7 | gate_40)')
print('H₁: p(ret7 | gate_30) ≠ p(ret7 | gate_40)')
print(f'\n[Chi-cuadrada]')
print(f'  χ² = {chi2_7:.4f}, gl = {dof_7}, p = {p_chi2_7:.4f}')
print(f'[Prueba Z de proporciones]')
print(f'  z  = {z7:.4f},            p = {p_z7:.4f}')
conclusion = 'Se rechaza H₀' if p_chi2_7 < alpha else 'No se rechaza H₀'
print(f'\nConclusión: {conclusion} (α = {alpha})')
print(f'\nTasas de retención día 7:')
print(f'  gate_30: {counts7[0]/nobs7[0]*100:.2f}%')
print(f'  gate_40: {counts7[1]/nobs7[1]*100:.2f}%')

**Interpretación:** Se rechaza la hipótesis nula (H₀).Existe evidencia estadísticamente significativa de que hay una diferencia en la tasa de retención al día 7 entre los dos grupos.



### Resumen de pruebas A/B

In [ ]:
resumen_ab = pd.DataFrame({
    'Variable'        : ['sum_gamerounds (t-test)', 'sum_gamerounds (Mann-Whitney)', 'retention_1 (chi²)', 'retention_7 (chi²)'],
    'Estadístico'     : [round(t_stat,4), round(u_stat,0), round(chi2_1,4), round(chi2_7,4)],
    'Valor p'         : [round(t_pval,4), round(u_pval,4), round(p_chi2_1,4), round(p_chi2_7,4)],
    'Significativo (α=0.05)': [t_pval<0.05, u_pval<0.05, p_chi2_1<0.05, p_chi2_7<0.05]
})
print('=== Resumen de Pruebas A/B ===')
display(resumen_ab)

---
## Análisis mediante regresión

### Regresión lineal

In [ ]:
# ── Regresión lineal ──────────────────────────────────────────────────────────
lm = smf.ols('sum_gamerounds ~ C(version, Treatment(reference="gate_30"))', data=df).fit()
print(lm.summary())

In [ ]:
# ── Interpretación del coeficiente ────────────────────────────────────────────
coef_lm = lm.params['C(version, Treatment(reference="gate_30"))[T.gate_40]']
pval_lm = lm.pvalues['C(version, Treatment(reference="gate_30"))[T.gate_40]']

print('=== Regresión Lineal: sum_gamerounds ~ version ===')
print(f'Coeficiente gate_40 : {coef_lm:.4f}')
print(f'Valor p             : {pval_lm:.4f}')
significativo = 'Sí' if pval_lm < 0.05 else 'No'
print(f'¿Significativo?     : {significativo} (α = 0.05)')
print(f'\nInterpretación: En promedio, los usuarios del grupo gate_40 juegan')
print(f'{coef_lm:+.2f} rondas que los de gate_30. Este efecto es{" " if significativo=="Sí" else " NO "}estadísticamente significativo.')

**Interpretación:** El signo negativo (-1.1575) indica que, en promedio, los jugadores del grupo gate_40 juegan aproximadamente 1.16 rondas menos que los del grupo gate_30, manteniendo todo lo demás constante.



### Regresión logística

In [ ]:
# ── Regresión logística para retention_1 ──────────────────────────────────────
logit1 = smf.logit('retention_1 ~ C(version, Treatment(reference="gate_30"))', data=df).fit()
print(logit1.summary())

In [ ]:
# ── Odds ratio e interpretación ───────────────────────────────────────────────
coef_l1 = logit1.params['C(version, Treatment(reference="gate_30"))[T.gate_40]']
pval_l1 = logit1.pvalues['C(version, Treatment(reference="gate_30"))[T.gate_40]']
or_l1   = np.exp(coef_l1)

print('=== Regresión Logística: retention_1 ~ version ===')
print(f'Coeficiente (log-odds) gate_40: {coef_l1:.4f}')
print(f'Odds Ratio gate_40            : {or_l1:.4f}')
print(f'Valor p                       : {pval_l1:.4f}')
significativo = 'Sí' if pval_l1 < 0.05 else 'No'
print(f'¿Significativo?               : {significativo} (α = 0.05)')
print(f'\nInterpretación: El odds de retención al día 1 en gate_40 es {or_l1:.4f} veces')
print(f'el del grupo gate_30. Esto corresponde a un cambio del {(or_l1-1)*100:+.2f}%.')

**Interpretación:** Odds Ratio de 0.9764 significa que los jugadores del grupo gate_40 tienen un 2.36% menos odds (probabilidad relativa) de regresar al día 1 comparado con los jugadores del grupo gate_30.
Como el OR < 1, indica que el grupo gate_40 tiene menor probabilidad de retención al día 1.

Mover el gate al nivel 40 está asociado con una ligera disminución en las probabilidades de que el jugador vuelva al día siguiente, aunque esta diferencia es pequeña.



### Regresión logística

In [ ]:
# ── Regresión logística para retention_7 ──────────────────────────────────────
logit7 = smf.logit('retention_7 ~ C(version, Treatment(reference="gate_30"))', data=df).fit()
print(logit7.summary())

In [ ]:
# ── Odds ratio e interpretación ───────────────────────────────────────────────
coef_l7 = logit7.params['C(version, Treatment(reference="gate_30"))[T.gate_40]']
pval_l7 = logit7.pvalues['C(version, Treatment(reference="gate_30"))[T.gate_40]']
or_l7   = np.exp(coef_l7)

print('=== Regresión Logística: retention_7 ~ version ===')
print(f'Coeficiente (log-odds) gate_40: {coef_l7:.4f}')
print(f'Odds Ratio gate_40            : {or_l7:.4f}')
print(f'Valor p                       : {pval_l7:.4f}')
significativo = 'Sí' if pval_l7 < 0.05 else 'No'
print(f'¿Significativo?               : {significativo} (α = 0.05)')
print(f'\nInterpretación: El odds de retención al día 7 en gate_40 es {or_l7:.4f} veces')
print(f'el del grupo gate_30. Esto corresponde a un cambio del {(or_l7-1)*100:+.2f}%.')

**Interpretación:** El Odds Ratio de 0.9473 significa que los jugadores del grupo gate_40 tienen un 5.27% menos odds de regresar al día 7 comparado con los jugadores del grupo gate_30.
Como el OR < 1, indica que el grupo gate_40 tiene menor probabilidad de retención a la semana.

Mover el gate del nivel 30 al nivel 40 está asociado con una reducción del 5.27% en las probabilidades de que el jugador siga jugando después de 7 días.



# Comparación: Pruebas A/B vs. Regresión

In [ ]:
# ── Tabla comparativa ─────────────────────────────────────────────────────────
comparacion = pd.DataFrame({
    'Variable'         : ['sum_gamerounds', 'retention_1', 'retention_7'],
    'Prueba A/B'       : ['Mann-Whitney U', 'Chi-cuadrada', 'Chi-cuadrada'],
    'p-valor (A/B)'    : [round(u_pval,4), round(p_chi2_1,4), round(p_chi2_7,4)],
    'Sig. A/B'         : [u_pval<0.05, p_chi2_1<0.05, p_chi2_7<0.05],
    'Modelo regresión' : ['OLS lineal', 'Logística', 'Logística'],
    'p-valor (regr.)'  : [round(pval_lm,4), round(pval_l1,4), round(pval_l7,4)],
    'Sig. regr.'       : [pval_lm<0.05, pval_l1<0.05, pval_l7<0.05],
    'Odds Ratio'       : ['N/A', round(or_l1,4), round(or_l7,4)]
})
print('=== Comparación: Pruebas A/B vs. Modelos de Regresión ===')
display(comparacion)
print('\nObservación: Cuando solo hay un predictor binario, las pruebas A/B y')
print('la regresión son equivalentes (misma información, misma conclusión).')

---
Preguntas

**1. ¿Los resultados sugieren que cambiar la puerta tuvo algún efecto sobre el comportamiento?**

Sí, aunque el efecto depende de qué variable miremos. Para las rondas jugadas (sum_gamerounds), la prueba Mann-Whitney dio un valor p significativo (< 0.05), lo que indica que sí hay diferencia estadística entre grupos, aunque en la práctica la diferencia en medianas es pequeña. Donde el efecto es más claro y relevante es en la retención a 7 días: la prueba chi-cuadrada mostró un p significativo, y la dirección de la diferencia favorece a gate_30, o sea, mantener la puerta en el nivel 30 retiene mejor a los jugadores a largo plazo. Para retención al día 1 el efecto también existe pero es más débil. En resumen: mover la puerta al nivel 40 no ayudó, y en retención a 7 días claramente perjudicó.

---

**2. ¿Las pruebas A/B y los modelos de regresión llevan a conclusiones similares?**

Sí, en la tabla comparativa los valores p de las pruebas A/B y los de los modelos de regresión son prácticamente idénticos, y las conclusiones coinciden en los tres casos. Eso tiene sentido: cuando solo tienes una variable explicativa binaria, hacer una prueba t o una regresión lineal es matemáticamente equivalente, igual que hacer chi-cuadrada o una regresión logística. La diferencia no está en el resultado, sino en la flexibilidad que te da cada enfoque.

---

**3. ¿Qué aporta el enfoque de regresión frente a una prueba A/B simple?**

Más allá de lo que ya se menciona en el trabajo, la regresión te permite hacer el análisis más honesto y completo. En la vida real los grupos A y B nunca son perfectamente comparables en todo; siempre hay ruido de otras variables. La regresión te deja "limpiar" ese ruido incluyendo controles. También es más interpretable para usuarios: decirle a alguien "el odds ratio es 0.93" comunica el tamaño del efecto, no solo si existe o no. Y si el experimento escala o se complica, la regresión escala con él sin necesidad de rediseñar el análisis desde cero.

---

**4. ¿En qué casos de analítica del marketing sería más útil la regresión?**

Cuando hay múltiples palancas al mismo tiempo, por ejemplo una campaña que varía el canal, el descuento ofrecido y el segmento de cliente a la vez. Con A/B puras se necesitaria analisis mas  complejos; con regresión metes todo en un modelo. También es clave donde quieres saber qué combinación de características del cliente predice mayor valor a futuro. O en modelos de desconexion, donde el objetivo no es comparar dos grupos sino identificar qué variables aumentan el riesgo de que un cliente se vaya.
---

**5. Si se contara con información adicional, ¿qué variables podrían incorporarse?**

Podrian ser: plataforma (iOS vs Android), país o región del usuario , días desde la instalación al momento del experimento, si el usuario ha hecho compras dentro del juego, y el nivel máximo alcanzado antes del experimento. Con esas variables la regresión pasaría de ser muy similar del A/B test a un modelo mas descriptivo.

---Conclusión final

El experimento muestra que mover la puerta del nivel 30 al 40 no fue una buena decisión. Aunque el número de rondas jugadas no cambia de forma dramática (la diferencia en medianas es pequeña,el indicador más importante, la retención a 7 días, sí se ve afectado negativamente: los usuarios de gate_40 tienen menor probabilidad de seguir jugando una semana después, con un odds ratio < 1 y un p-valor significativo en la regresión logística. Esto sugiere que la puerta temprana actúa como un mecanismo de "enganche forzado" que, paradójicamente, hace que los jugadores valoren más el progreso y vuelvan al juego.
En cuanto a la relación entre A/B Testing y regresión, este ejercicio deja claro que no son enfoques opuestos sino complementarios. Con un diseño experimental limpio y una sola variable de tratamiento, ambos llevan exactamente a la misma conclusión. La regresión es mejor cuando el entorno es más complejo: múltiples variables, segmentos diferentes o necesidad de cuantificar el tamaño del efecto de forma interpretable. En analítica de marketing lo ideal es dominar ambos: el A/B test para diseñar experimentos rigurosos, y la regresión para extraer todo el valor de los datos resultantes.

En un experimento simple con dos grupos, una prueba A/B y una regresión con variable indicadora responden preguntas equivalentes. La ventaja de la regresión es su flexibilidad para escenarios más complejos y realistas en analítica del marketing.